# Python Memory Management — Where Your Objects Live

> **Goal:** understand objects, references, copying, garbage collection, profiling, and practical ways to use less memory.

## Explain it like I am 5

Computer memory is a playroom. Python creates toys (objects) in the room and gives you sticky notes (variable names) that point to them. Several sticky notes can point to the same toy. When nobody can reach a toy anymore, Python tidies it away.

This notebook preserves the original reference-counting, `gc`, circular-reference, generator, and `tracemalloc` examples while making them safer and more complete.

## Learning map

| Level | Topics |
|---|---|
| Basic | objects, references, identity, mutability, `del` |
| Intermediate | reference counts, garbage collection, copies, object lifecycle |
| Advanced | weak references, `tracemalloc`, leak patterns, optimization |

> Most implementation details here describe **CPython**, the standard Python implementation. Other Python implementations may manage memory differently.

## 1. The big picture: stack and heap

This is a useful mental model, not a promise about every internal byte.

| Area | Tiny explanation |
|---|---|
| Call stack | active function-call frames: local names, where execution should return |
| Private heap | Python-managed area where objects usually live |
| Allocator | CPython's machinery that requests/reuses blocks of memory |

A local variable is usually a **reference stored in a frame**, while the list/dictionary/class instance it refers to is a Python object managed on the heap. Python handles allocation automatically; you do not manually `malloc` or `free` normal objects.

In [1]:
def make_box():
    local_name = ["ball", "car"]
    return local_name

toy_box = make_box()
print(toy_box)
print("The function frame ended, but the returned list is still reachable.")

['ball', 'car']
The function frame ended, but the returned list is still reachable.


## 2. Objects, names, references, and `id()`

Assignment does not copy an object. It makes another name point to the same object. `id(object)` is unique for that object's lifetime. In CPython it often resembles a memory address, but portable code must treat it only as an identity token.

In [2]:
box_a = ["red block"]
box_b = box_a

print("Same object?", box_a is box_b)
print("Same id?", id(box_a) == id(box_b))
box_b.append("blue block")
print("box_a changed too:", box_a)

Same object? True
Same id? True
box_a changed too: ['red block', 'blue block']


### `is` versus `==`

- `a == b` asks, “Do these values compare equal?”
- `a is b` asks, “Are these the exact same object?”

Use `is None`, because `None` is a singleton. Do not use `is` to compare numbers or strings; interning is an optimization and can make identity appear inconsistent.

In [3]:
first = [1, 2]
second = [1, 2]
print("Equal values:", first == second)
print("Same object:", first is second)
print("None check:", None is None)

Equal values: True
Same object: False
None check: True


## 3. Reference counting — the first tidy-up system

CPython counts strong references to most objects. When the count reaches zero, the object can usually be reclaimed immediately.

The original lesson used `sys.getrefcount()`. Its result includes one temporary reference created by the function call, and notebooks/debuggers may hold extra references, so use it for learning—not exact production accounting.

In [4]:
import sys

a = []
print("After a = []:", sys.getrefcount(a))
b = a
print("After b = a:", sys.getrefcount(a))
del b
print("After del b:", sys.getrefcount(a))

After a = []: 2
After b = a: 3
After del b: 2


## 4. `del` removes a name, not necessarily an object

`del name` peels off one sticky note. If another reference exists, the object remains alive. `del items[0]` asks a container to remove an item; it still does not guarantee immediate operating-system memory release.

In [5]:
original = {"snack": "apple"}
alias = original
object_id = id(original)
del original
print("Still alive through alias:", alias)
print("Same identity:", id(alias) == object_id)

Still alive through alias: {'snack': 'apple'}
Same identity: True


## 5. Mutable versus immutable objects

| Usually mutable | Usually immutable |
|---|---|
| `list`, `dict`, `set`, `bytearray` | `int`, `float`, `bool`, `str`, `bytes`, `tuple`, `frozenset` |

Mutable means the same object can change. Immutable means an operation produces another object. A tuple is immutable, but it may contain a mutable object that can change.

In [6]:
number = 10
old_number_id = id(number)
number += 1
print("Integer got a new identity:", id(number) != old_number_id)

nested = ("fixed label", [1, 2])
nested[1].append(3)
print("Tuple still exists; its inner list changed:", nested)

Integer got a new identity: True
Tuple still exists; its inner list changed: ('fixed label', [1, 2, 3])


## 6. Shallow copy versus deep copy

Imagine folders inside a schoolbag.

- A **shallow copy** makes a new schoolbag but reuses the same inner folders.
- A **deep copy** recursively makes new folders too.

Deep copying can be expensive, may copy more than needed, and cannot sensibly duplicate every resource (open files, sockets, modules). Design explicit copy behavior for complex classes.

In [7]:
import copy

original = [["A"], ["B"]]
shallow = copy.copy(original)
deep = copy.deepcopy(original)

original[0].append("changed")
print("Original:", original)
print("Shallow shares inner list:", shallow)
print("Deep has independent inner list:", deep)

Original: [['A', 'changed'], ['B']]
Shallow shares inner list: [['A', 'changed'], ['B']]
Deep has independent inner list: [['A'], ['B']]


### Common nested-list mistake

`[[0] * 3] * 4` repeats the **same inner list reference**. A comprehension creates separate rows.

In [8]:
wrong_grid = [[0] * 3] * 4
wrong_grid[0][0] = 9

right_grid = [[0] * 3 for _ in range(4)]
right_grid[0][0] = 9

print("Shared rows:", wrong_grid)
print("Independent rows:", right_grid)

Shared rows: [[9, 0, 0], [9, 0, 0], [9, 0, 0], [9, 0, 0]]
Independent rows: [[9, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0]]


## 7. Circular references and cyclic garbage collection

Reference counting alone cannot clean a loop: object A points to B, and B points back to A. CPython's cyclic garbage collector periodically looks for unreachable container cycles.

The original `MyObject` example is preserved below. We use `weakref.finalize` for a predictable teaching signal instead of relying on `__del__` timing.

In [9]:
import gc
import weakref

class MyObject:
    def __init__(self, name):
        self.name = name
        print(f"Object {self.name} created")


finalized = []
obj1 = MyObject("obj1")
obj2 = MyObject("obj2")
weakref.finalize(obj1, finalized.append, "obj1 cleaned")
weakref.finalize(obj2, finalized.append, "obj2 cleaned")
obj1.ref = obj2
obj2.ref = obj1

del obj1
del obj2
collected = gc.collect()
print("Collector found objects:", collected)
print("Finalizers called:", sorted(finalized))

Object obj1 created
Object obj2 created
Collector found objects: 283
Finalizers called: ['obj1 cleaned', 'obj2 cleaned']


## 8. The `gc` module — inspect and tune carefully

The original notebook enabled, disabled, collected, printed stats, and inspected `gc.garbage`. Those tools remain here.

| Function | Purpose |
|---|---|
| `gc.enable()` / `gc.disable()` | turn cyclic collection on/off |
| `gc.isenabled()` | check status |
| `gc.collect()` | request a collection; returns number found |
| `gc.get_count()` | allocation counters since collections |
| `gc.get_threshold()` | collection thresholds |
| `gc.get_stats()` | per-generation statistics |
| `gc.garbage` | uncollectable/debug-retained objects; usually empty |

Disabling GC does **not** disable reference counting in CPython. Always restore prior state. Manual collection in a hot loop usually hurts performance.

In [10]:
was_enabled = gc.isenabled()
gc.disable()
print("Enabled after disable?", gc.isenabled())
if was_enabled:
    gc.enable()

print("Enabled now?", gc.isenabled())
print("Thresholds:", gc.get_threshold())
print("Counts:", gc.get_count())
print("Generations reported:", len(gc.get_stats()))
print("Uncollectable/debug-retained objects:", len(gc.garbage))
print("Manual collection found:", gc.collect())

Enabled after disable? False
Enabled now? True
Thresholds: (2000, 10, 10)
Counts: (326, 0, 0)
Generations reported: 3
Uncollectable/debug-retained objects: 0
Manual collection found: 0


## 9. Object lifecycle and cleanup

Typical story: allocate → initialize → use → lose last strong reference → finalize/reclaim.

Do not depend on `__del__` for important resources. Its timing varies, interpreter shutdown is awkward, and resurrection is possible. Use context managers for deterministic cleanup.

In [11]:
from contextlib import contextmanager

@contextmanager
def pretend_resource(name):
    print("OPEN", name)
    try:
        yield name
    finally:
        print("CLOSE", name)

with pretend_resource("tiny-file") as resource:
    print("USE", resource)

OPEN tiny-file
USE tiny-file
CLOSE tiny-file


## 10. Weak references

A weak reference observes an object without keeping it alive. Useful for caches, parent links, and registries. Not every built-in type supports weak references; custom class instances usually do.

In [12]:
class Child:
    pass

child = Child()
observer = weakref.ref(child)
print("Before del:", observer() is child)
del child
gc.collect()
print("After del:", observer())

Before del: True
After del: None


## 11. `sys.getsizeof()` is shallow

It reports the direct size of one object, not all objects it refers to. Exact numbers depend on Python build, platform, and version. Treat them as measurements from this machine, not universal constants.

In [13]:
empty_list = []
nested_list = [[1, 2, 3], [4, 5, 6]]
print("Empty list bytes:", sys.getsizeof(empty_list))
print("Outer nested list bytes only:", sys.getsizeof(nested_list))
print("Inner lists separately:", [sys.getsizeof(item) for item in nested_list])

Empty list bytes: 56
Outer nested list bytes only: 72
Inner lists separately: [88, 88]


### A teaching-only recursive size helper

Avoid double-counting shared objects by remembering identities. Production profilers are better for complex graphs, extension objects, NumPy buffers, and allocator details.

In [14]:
def deep_size(obj, seen=None):
    if seen is None:
        seen = set()
    object_id = id(obj)
    if object_id in seen:
        return 0
    seen.add(object_id)

    size = sys.getsizeof(obj)
    if isinstance(obj, dict):
        size += sum(deep_size(k, seen) + deep_size(v, seen) for k, v in obj.items())
    elif isinstance(obj, (list, tuple, set, frozenset)):
        size += sum(deep_size(item, seen) for item in obj)
    return size

shared = [1, 2, 3]
structure = {"left": shared, "right": shared}
print("Shallow bytes:", sys.getsizeof(structure))
print("Approximate deep bytes:", deep_size(structure))

Shallow bytes: 184
Approximate deep bytes: 447


## 12. Generators versus lists

A list cooks and stores the whole meal. A generator cooks one bite when requested. This reduces peak memory for streaming pipelines but generators are one-pass and compute lazily.

The original generator example produced numbers and stopped after 10; it is preserved with bounded output.

In [15]:
def generate_numbers(n):
    for number in range(n):
        yield number

for number in generate_numbers(100_000):
    print(number, end=" ")
    if number > 10:
        break
print()

number_list = [number for number in range(100_000)]
number_generator = (number for number in range(100_000))
print("List container bytes:", sys.getsizeof(number_list))
print("Generator object bytes:", sys.getsizeof(number_generator))

0 1 2 3 4 5 6 7 8 9 10 11 
List container bytes: 800984
Generator object bytes: 200


### Generator trade-offs

- Great for files, database rows, transformations, and large sequences.
- Usually consumed once; a second loop sees nothing unless recreated.
- `len(generator)` and indexing do not work.
- Holding yielded items elsewhere can erase the memory benefit.
- A generator can keep its frame and referenced objects alive until exhausted or closed.

In [16]:
one_pass = (value * value for value in range(3))
print("First pass:", list(one_pass))
print("Second pass:", list(one_pass))

First pass: [0, 1, 4]
Second pass: []


## 13. Profiling allocations with `tracemalloc`

`tracemalloc` traces Python memory allocations. The original lesson took a snapshot after building a list. Here we retain the list long enough to appear, bound the output, then compare before/after snapshots.

In [17]:
import tracemalloc

def create_list():
    return [number for number in range(10_000)]

tracemalloc.start()
before = tracemalloc.take_snapshot()
profiled_list = create_list()
after = tracemalloc.take_snapshot()

print("Top allocation differences:")
for statistic in after.compare_to(before, "lineno")[:3]:
    print(statistic)

current, peak = tracemalloc.get_traced_memory()
print(f"Current traced: {current:,} bytes | peak: {peak:,} bytes")
del profiled_list
tracemalloc.stop()

Top allocation differences:


C:\Users\HP\AppData\Local\Temp\ipykernel_22576\1985479542.py:4: size=385 KiB (+385 KiB), count=9658 (+9658), average=41 B
C:\Users\HP\anaconda3\Lib\tracemalloc.py:560: size=328 B (+328 B), count=1 (+1), average=328 B
C:\Users\HP\anaconda3\Lib\tracemalloc.py:423: size=328 B (+328 B), count=1 (+1), average=328 B
Current traced: 409,042 bytes | peak: 429,310 bytes


## 14. What people call a “memory leak” in Python

Often memory is not lost; objects are still reachable accidentally.

Common causes:

- an ever-growing global list/dictionary/cache;
- callbacks or event listeners never unregistered;
- closures retaining large objects;
- thread-local data kept by long-lived threads;
- unclosed resources or native-extension allocations;
- tasks/results accumulated faster than consumers process them;
- allocator fragmentation/high-water behavior: freed memory may stay reserved for reuse rather than return immediately to the OS.

In [18]:
from functools import lru_cache

@lru_cache(maxsize=128)
def bounded_square(number):
    return number * number

for value in range(500):
    bounded_square(value)

print(bounded_square.cache_info())
bounded_square.cache_clear()

CacheInfo(hits=0, misses=500, maxsize=128, currsize=128)


## 15. Practical optimization toolbox

| Situation | Consider |
|---|---|
| huge list used once | generator/iterator |
| reading a large file | line iteration or chunks |
| millions of uniform numbers | `array`, NumPy, or packed/binary formats |
| many simple class instances | `dataclass(slots=True)` / `__slots__` |
| repeated immutable data | reuse/intern deliberately, normalize data |
| unbounded cache | `lru_cache(maxsize=...)`, TTL/eviction |
| copying nested structures | avoid copy, copy only changed parts |
| unknown growth | measure with `tracemalloc`, process metrics, profiler |

Optimize after measuring. Smaller code is not automatically smaller memory, and `sys.getsizeof()` alone is not a full profiler.

In [19]:
class RegularPoint:
    def __init__(self, x, y):
        self.x = x
        self.y = y

class SlottedPoint:
    __slots__ = ("x", "y")
    def __init__(self, x, y):
        self.x = x
        self.y = y

regular = RegularPoint(1, 2)
slotted = SlottedPoint(1, 2)
print("Regular instance + __dict__:", sys.getsizeof(regular) + sys.getsizeof(regular.__dict__))
print("Slotted instance:", sys.getsizeof(slotted))

Regular instance + __dict__: 344
Slotted instance: 48


## 16. Common memory mistakes

| Mistake | Better habit |
|---|---|
| assuming assignment copies | know when references alias |
| using `is` for value comparison | use `==`; reserve `is` for identity/singletons |
| repeating nested lists with `*` | use a comprehension |
| deep-copying everything | copy only what must be independent |
| relying on `del` to free OS memory | remove all references; understand allocator reuse |
| disabling GC and forgetting it | restore prior state; tune only with evidence |
| unbounded caches/history | cap, expire, or stream |
| materializing a whole pipeline | use iterators/chunks |
| trusting one size number | profile over time and inspect object graphs |
| important cleanup in `__del__` | use context managers/finalizers |

# Quick Revision Cheat Sheet

```python
id(obj)                    # identity token during object's lifetime
sys.getrefcount(obj)       # CPython teaching aid; includes temporary reference
sys.getsizeof(obj)         # shallow size only
del name                   # delete one reference/name
copy.copy(obj)             # new outer object; nested references shared
copy.deepcopy(obj)         # recursively copy supported object graph
gc.collect()               # request cyclic collection
weakref.ref(obj)           # observe without keeping alive
tracemalloc.start()        # trace Python allocations
```

## Remember

- Names point to objects; assignment normally does not copy.
- Mutable objects can change in place; immutable operations make new values.
- CPython uses reference counting plus a cyclic collector.
- `del` removes a reference, not necessarily the object or process memory.
- Generators reduce peak memory when data can be streamed.
- Most “leaks” are unintended long-lived references; measure before optimizing.